# Phase 0 pilot: orchestrated logging + fold-over ground truth

Runs the Phase 0 pilot (README's "Phase 0 split" section): a mix of natural-retrieval
logging episodes and fold-over ground-truth pairs for the 5 pre-registered memories,
resumable across Kaggle sessions via a Kaggle Dataset checkpoint.

**Built for Kaggle's "Save & Run All"**: every cell below runs unconditionally, in
order, with no manual choices -- the only thing to edit is `DATASET_SLUG` in the
settings cell (once, to your own Kaggle username/dataset), before the first run.

**Always use Save & Run All for every Phase 0 session -- never run cells
interactively one at a time.** An interactive session that's idle or not being
actively watched can be torn down by Kaggle without warning; if that happens before
the first checkpoint, everything is lost (this happened for real: session 1 ran 1h+
with both GPUs busy and was lost entirely, because the old schedule only checkpointed
every 2 hours and hadn't pushed anything yet). Save & Run All runs unattended to
completion and isn't subject to that failure mode.

## Settings to click before running anything

- **Accelerator: GPU T4 x2** (Notebook settings, right sidebar) -- both GPUs are used by default (`WORKERS = 2`); run `multi_gpu_sanity_check.ipynb` once first to confirm both are actually being used before committing to a full session.
- **Internet: ON** (same sidebar) -- needed for pip installs and downloads.
- **Kaggle API token**: open **Add-ons -> Secrets**, add a secret labeled exactly `KAGGLE_API_TOKEN` (format `KGAT_...`, from kaggle.com Settings -> Account -> API tokens), and toggle it on for this notebook -- required for the checkpoint Dataset push/pull below.
- **Add Input -> your checkpoint Dataset**: on the very first run this dataset won't
  exist yet, so there's nothing to attach -- `run_phase0_pilot.py` detects that and
  starts fresh, printing "assuming this is the FIRST session." After the first run
  completes (it creates the dataset via the `kaggle` CLI automatically, within
  minutes of starting -- see "Checkpoint interval" below), attach that same dataset
  as an input on every later session so it can resume.
- **Time budget**: the run cell below uses a 10.5-hour budget by default, under
  Kaggle's 12-hour session cap -- edit `TIME_BUDGET_SECONDS` if you want a shorter
  session (e.g. for a quick test).
- **Checkpoint interval**: an initial checkpoint (job list only, no episodes yet) is
  pushed within minutes of starting, before anything else runs -- proves the push
  path works at the very start of every session instead of only after the first
  full chunk. After that, the first chunk is 30 minutes
  (`FIRST_CHECKPOINT_INTERVAL_SECONDS`) so the first checkpoint with actual episode
  results also lands early, then every chunk after that is 1 hour
  (`CHECKPOINT_INTERVAL_SECONDS`) -- a crash loses at most about that much work, not
  the whole session.

## What happens each session

1. Copies in the last checkpoint (if any).
2. Generates the job list on the very first session only (same file reused after),
   and immediately pushes an initial checkpoint containing just that job list.
3. Runs jobs (logging + ground-truth fold pairs) until the time budget or the job
   list is exhausted, whichever comes first -- the first chunk is 30 minutes, every
   chunk after that is 1 hour, each followed by a checkpoint push. Each worker
   prints one line per finished episode as it happens, so the notebook is never
   silent for long.
4. Writes the pre-registered validation metrics against whatever's done so far.
5. Pushes the updated checkpoint back out as a new Dataset version.

Run this notebook (**Save & Run All -- never an interactive session**) once per
Kaggle session until the job list is exhausted -- check the last cell's printed
summary to see how much is left.

## Settings -- edit DATASET_SLUG once, before the first run

In [ ]:
DATASET_SLUG = "your-kaggle-username/memory-pilot-phase0-checkpoints"  # EDIT THIS
N_LOGGING = 1050       # README's "Phase 0 split" section: ~70% of ~1500 episodes
N_GT_PAIRS = 225       # ~30% of ~1500 episodes, as fold-over pairs (2 episodes/pair)
TIME_BUDGET_SECONDS = 37_800  # 10.5h, under Kaggle's 12h session cap (time guard)
FIRST_CHECKPOINT_INTERVAL_SECONDS = 1_800  # 30 min -- gets the first real (post-job-list) checkpoint out fast
CHECKPOINT_INTERVAL_SECONDS = 3_600  # push to the checkpoint Dataset every ~1h after that, not just at the end
WORKERS = 2  # one per GPU (T4 x2) -- run multi_gpu_sanity_check.ipynb once before the first real session


In [ ]:
if 'your-kaggle-username' in DATASET_SLUG:
    raise SystemExit(
        'FAIL: DATASET_SLUG is still the placeholder value -- edit the settings cell above '
        "to your own '<kaggle-username>/<dataset-name>' before running this notebook."
    )
print('DATASET_SLUG looks real:', DATASET_SLUG)


## 0. Setup

In [ ]:
import os
# Standard fix for the PyTorch CUDA caching allocator's OOM-from-fragmentation
# failure mode -- set before anything imports torch. See README.md's OOM section.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
print('GPU:')
subprocess.run(['nvidia-smi'])


In [ ]:
!rm -rf memory-pilot
!git clone https://github.com/ibo-agentic/memory-pilot.git
%cd memory-pilot
!git rev-parse HEAD


In [ ]:
import subprocess
import sys

def _run(cmd):
    """Runs cmd and raises loudly on failure -- `!pip install ...` in a
    Jupyter cell does NOT stop the notebook on failure (it just prints
    whatever pip printed and moves on), which is exactly how a real Kaggle
    session's broken install was only discovered several cells later, at the
    import-check cell, with no idea why. Also uses sys.executable (THIS
    kernel's own Python) rather than relying on `!pip`/`!python` resolving
    to the same interpreter the kernel runs under -- on some Kaggle images
    they are NOT the same, which silently installs into the wrong
    environment and is a strong candidate for exactly the
    "No module named 'memory_ope'" failure seen on a real session despite
    the install cell appearing to succeed."""
    print('+ ' + ' '.join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise SystemExit(f'FAIL: command exited {result.returncode}: {" ".join(cmd)}')

_run([sys.executable, '-m', 'pip', 'install', 'alfworld', 'tqdm', 'termcolor', 'sentence-transformers', '--quiet'])
_run([sys.executable, '-m', 'pip', 'install', 'transformers', 'accelerate', 'bitsandbytes', '--quiet'])
_run(['alfworld-download'])

# Editable-installs both packages so alfworld_pilot and memory_ope import in
# EVERY process (including run_jobs.py's own subprocess workers) with no
# PYTHONPATH needed anywhere -- verified locally, see README's Orchestration
# section. --quiet deliberately dropped for these two (unlike the installs
# above) so a real failure here is visible in the notebook's own output
# instead of being silently swallowed -- this is the install that actually
# failed on a real Kaggle session once.
_run([sys.executable, '-m', 'pip', 'install', '-e', '.'])
_run([sys.executable, '-m', 'pip', 'install', '-e', 'alfworld_pilot'])
_run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kaggle', '--quiet'])


In [ ]:
import os

# Kaggle's newer single-token API auth (KAGGLE_API_TOKEN, format KGAT_...)
# replaces the old kaggle.json username/key pair. Read it from Kaggle
# Secrets -- never hardcode it, never print it.
try:
    from kaggle_secrets import UserSecretsClient
    _kaggle_api_token = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception as e:
    raise SystemExit(
        'FAIL: could not read the KAGGLE_API_TOKEN secret -- add it via this '
        'notebook\'s Add-ons -> Secrets menu (label exactly KAGGLE_API_TOKEN, '
        f'value starting KGAT_...) before running this cell. Error: {e}'
    )

os.environ['KAGGLE_API_TOKEN'] = _kaggle_api_token
del _kaggle_api_token  # don't leave the token sitting in a notebook-visible variable
print('KAGGLE_API_TOKEN loaded from Kaggle Secrets and set as an environment variable (value not printed).')


In [ ]:
import sys, pathlib

# Belt-and-suspenders fix (2026-10-02), after a real fresh Kaggle session
# still failed with "No module named 'memory_ope'" even with the previous
# version of this cell (a site.addsitedir refresh) in place. Rather than
# depend on pip's editable-install machinery and Kaggle's exact
# site-packages layout working correctly -- cell 7 now at least installs
# with THIS kernel's own sys.executable, closing one likely gap -- insert
# both packages' src/ directories onto sys.path DIRECTLY, computed from the
# current working directory (cell 6 already `%cd memory-pilot`'d into the
# repo root). This is fully deterministic and doesn't depend on ANY pip/
# site-packages internals, so it keeps working even if a future Kaggle
# image breaks editable installs in some new way.
REPO_ROOT = pathlib.Path.cwd()
for _src_dir in (REPO_ROOT / 'src', REPO_ROOT / 'alfworld_pilot' / 'src'):
    if _src_dir.is_dir():
        if str(_src_dir) not in sys.path:
            sys.path.insert(0, str(_src_dir))
    else:
        print(f'WARNING: expected source directory not found: {_src_dir} (cwd={REPO_ROOT})')

for _mod_name in list(sys.modules):
    if _mod_name == 'alfworld_pilot' or _mod_name.startswith('alfworld_pilot.') or _mod_name == 'memory_ope' or _mod_name.startswith('memory_ope.'):
        del sys.modules[_mod_name]

print('sys.path now starts with:', sys.path[:3])


In [ ]:
try:
    import alfworld_pilot
    import memory_ope
    from alfworld_pilot import job_list, run_jobs, run_estimators, run_phase0_pilot  # noqa: F401
    print('Import check PASSED:')
    print('  alfworld_pilot ->', alfworld_pilot.__file__)
    print('  memory_ope     ->', memory_ope.__file__)
except ImportError as e:
    # If this ever fails again despite the direct sys.path fix above, dump
    # enough state here to actually diagnose it next time, instead of
    # guessing again from scratch.
    import subprocess, sys as _sys
    print('sys.path:', _sys.path)
    subprocess.run([_sys.executable, '-m', 'pip', 'show', 'alfworld-pilot'])
    subprocess.run([_sys.executable, '-m', 'pip', 'show', 'memory-ope'])
    raise SystemExit(f'Import check FAILED: {e!r} -- see sys.path/pip show output above for diagnosis.')


## 1. Run the pilot for this session

Idempotent and resumable: safe to run this exact cell again in a brand-new session
(after re-attaching the checkpoint Dataset as input) -- it picks up exactly where the
last session's Dataset version left off, per job_id, never re-running or skipping a
job incorrectly (see `tests/test_job_list_and_runner.py` and the handoff test).

In [ ]:
%cd alfworld_pilot
import sys
sys.argv = [
    'run_phase0_pilot',
    '--dataset-slug', DATASET_SLUG,
    '--n-logging', str(N_LOGGING),
    '--n-gt-pairs', str(N_GT_PAIRS),
    '--time-budget-seconds', str(TIME_BUDGET_SECONDS),
    '--first-checkpoint-interval-seconds', str(FIRST_CHECKPOINT_INTERVAL_SECONDS),
    '--checkpoint-interval-seconds', str(CHECKPOINT_INTERVAL_SECONDS),
    '--workers', str(WORKERS),
]
from alfworld_pilot import run_phase0_pilot
run_phase0_pilot.main()


## 2. Session summary

In [ ]:
import json, pathlib

results_path = pathlib.Path('/kaggle/working/logs_kaggle/phase0_results.jsonl')
job_list_path = pathlib.Path('/kaggle/working/logs_kaggle/phase0_job_list.json')

if job_list_path.exists():
    jobs = json.load(open(job_list_path))
    n_total = len(jobs)
else:
    n_total = 0

if results_path.exists():
    with open(results_path) as f:
        n_done = sum(1 for line in f if line.strip())
else:
    n_done = 0

print(f'Progress: {n_done}/{n_total} jobs done ({100 * n_done / n_total:.1f}%)' if n_total else 'No job list found.')
if n_total and n_done < n_total:
    print('Not finished -- attach the checkpoint Dataset this session just pushed as input on the next session, and Save & Run All again.')
elif n_total:
    print('All jobs done. See phase0_validation.json (also pushed to the checkpoint Dataset) for the validation metrics.')
